In [117]:
import pandas as pd
import numpy as np
import pickle
import random
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [118]:
df = pd.read_excel("DATASET_PHS.xlsx")
df = df.dropna()
df.head()

,No,Penyakit,Pertanyaan,Jawaban
0,1,Demam,Apa yang harus saya lakukan jika mengalami demam?,"Perbanyak cairan dan makanan bergizi, Minum ba..."
1,2,Demam,Bagaimana cara mengatasi demam?,"Perbanyak cairan dan makanan bergizi, Minum ba..."
2,3,Demam,Bagaimana cara menurunkan demam?,"Perbanyak cairan dan makanan bergizi, Minum ba..."
3,4,Demam,Bagaimana peran pola hidup sehat dalam mengata...,"Perbanyak cairan dan makanan bergizi, Minum ba..."
4,5,Demam,Cara menurunkan demam?,"Perbanyak cairan dan makanan bergizi, Minum ba..."


In [119]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['Pertanyaan'] = df['Pertanyaan'].astype(str).apply(clean_text)


In [120]:
def klasifikasi_medis(text):
    text = text.lower()

    # 1. Neurologi
    if any(k in text for k in ["pusing", "vertigo", "migrain", "sakit kepala", "kepala berat", "tremor", "epilepsi"]):
        return "neurologi"

    # 2. Kardiovaskular
    elif any(k in text for k in ["hipertensi", "jantung", "kolesterol"]):
        return "kardiovaskular"

    # 3. Pernapasan
    elif any(k in text for k in ["batuk", "pilek", "flu", "sesak nafas", "asma", "sinus", "rhinitis", "pernapasan"]):
        return "pernapasan"

    # 4. Pencernaan
    elif any(k in text for k in ["lambung", "maag", "perut", "mual", "muntah", "diare", "mencret", "sembelit"]):
        return "pencernaan"

    # 5. Metabolik & Darah
    elif any(k in text for k in ["diabetes", "anemia", "obesitas", "asam urat", "vitamin", "malnutrisi"]):
        return "metabolik"

    # 6. Kulit & Alergi
    elif any(k in text for k in ["gatal", "kurap", "alergi", "bengkak", "cacar", "campak", "biang keringat"]):
        return "kulit_alergi"

    # 7. Mata
    elif "mata" in text:
        return "mata"

    # 8. THT
    elif any(k in text for k in ["telinga", "tenggorokan", "amandel", "menelan", "hidung"]):
        return "tht"

    # 9. Otot & Tulang
    elif any(k in text for k in ["pinggang", "punggung", "leher", "otot", "badan", "kaki", "kram", "saraf"]):
        return "muskuloskeletal"

    # 10. Mental
    elif any(k in text for k in ["insomnia", "stres", "panik", "gangguan tidur", "kelelahan"]):
        return "mental"

    # 11. Reproduksi
    elif any(k in text for k in ["pcos", "kista", "menstruasi", "keputihan", "mens"]):
        return "reproduksi"

    # 12. Kemih
    elif any(k in text for k in ["kencing", "buang air kecil"]):
        return "urologi"

    # 13. Infeksi
    elif any(k in text for k in ["demam", "infeksi", "tifus", "dbd", "hepatitis"]):
        return "infeksi"

    # 14. Umum
    elif any(k in text for k in ["dehidrasi", "hipotermia", "nafsu makan"]):
        return "umum"

    else:
        return "tidak diketahui"

In [121]:
df['Penyakit'] = df['Penyakit'].apply(klasifikasi_medis)
df = df[df['Penyakit'].notna()]


In [122]:
print(df['Penyakit'].value_counts())


Penyakit
pencernaan         75
tidak diketahui    70
kulit_alergi       65
pernapasan         60
neurologi          40
muskuloskeletal    40
infeksi            36
tht                35
metabolik          30
mental             25
kardiovaskular     20
reproduksi         20
umum               15
mata               10
urologi             5
Name: count, dtype: int64


In [123]:
texts = df['Pertanyaan']
labels = df['Penyakit']
responses = df['Jawaban']


In [124]:
qa_dict = {}

for q, a in zip(texts, responses):
    qa_dict[q.strip()] = a

with open("qa_dict.pkl", "wb") as f:
    pickle.dump(qa_dict, f)


In [125]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)


In [126]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, padding='post')

In [127]:
X_train, X_test, y_train, y_test = train_test_split(
    padded, y, test_size=0.2, random_state=42
)


In [128]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D, Input

model = Sequential([
    Input(shape=(padded.shape[1],)),  # 🔥 penting

    Embedding(input_dim=5000, output_dim=128),

    SpatialDropout1D(0.3),

    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    Bidirectional(LSTM(64)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(set(y)), activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 10, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_8             │ (None, 10, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_16                │ (None, 10, 256)        │       263,168 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_17                │ (None, 128)            │       164,352 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,076,751 (4.11 MB)

 Trainable params: 1,076,751 (4.11 MB)

 Non-trainable params: 0 (0.00 B)

In [129]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

In [130]:
model.fit(X_train, y_train, batch_size=8, epochs=150)

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n=== HASIL EVALUASI ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))


Epoch 1/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.1055 - loss: 2.6280
Epoch 2/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.1376 - loss: 2.5435
Epoch 3/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.1628 - loss: 2.2672
Epoch 4/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.3716 - loss: 1.7904
Epoch 5/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.4587 - loss: 1.5114
Epoch 6/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.5963 - loss: 1.1545
Epoch 7/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.6261 - loss: 1.1534
Epoch 8/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.7110 - loss: 0.7882
Epoch 9/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.7775 - loss: 0.5818
Epoch 10/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.8257 - loss: 0.5346
Epoch 11/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.8647 - loss: 0.4327
Epoch 12/150
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/ste

In [131]:
model.save("chatbot_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)


In [132]:
response_dict = {}

for label, response in zip(labels, responses):
    if label not in response_dict:
        response_dict[label] = []
    response_dict[label].append(response)

with open("responses.pkl", "wb") as f:
    pickle.dump(response_dict, f)


In [133]:
model = load_model("chatbot_model.h5")
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))
label_encoder = pickle.load(open("label_encoder.pkl", "rb"))
responses = pickle.load(open("responses.pkl", "rb"))
qa_dict = pickle.load(open("qa_dict.pkl", "rb"))


In [134]:
def chatbot_response(text):
    text_clean = text.lower().strip()
    text_clean = re.sub(r'[^a-zA-Z\s]', '', text_clean)

    # =====================
    # 1. EXACT MATCH
    # =====================
    if text_clean in qa_dict:
        return qa_dict[text_clean]

    # =====================
    # 2. RULE BASED
    # =====================
    if "batuk" in text_clean:
        return "Untuk meredakan batuk, minum air hangat dan istirahat cukup."

    if "demam" in text_clean:
        return "Untuk menurunkan demam, istirahat cukup dan minum banyak air."

    if "mata" in text_clean and "gatal" in text_clean:
        return "Gunakan obat tetes mata dan hindari mengucek mata."

    # =====================
    # 3. MODEL LSTM
    # =====================
    seq = tokenizer.texts_to_sequences([text_clean])
    padded = pad_sequences(seq, maxlen=model.input_shape[1], padding='post')

    pred = model.predict(padded)
    confidence = np.max(pred)
    idx = np.argmax(pred)

    label = label_encoder.inverse_transform([idx])[0]

    print("DEBUG → Prediksi:", label)
    print("DEBUG → Confidence:", confidence)

    if confidence < 0.5:
        return "Maaf, saya belum memahami pertanyaan Anda."

    if label in responses:
        return random.choice(responses[label])

    return "Maaf, jawaban tidak ditemukan."


In [135]:
print("🤖 Chatbot Kesehatan")
print("Ketik 'exit' untuk keluar\n")

while True:
    user_input = input("Anda: ")

    if user_input.lower() == "exit":
        print("Bot: Terima kasih, semoga sehat selalu!")
        break

    print("Bot:", chatbot_response(user_input))


🤖 Chatbot Kesehatan
Ketik 'exit' untuk keluar

Anda: cara meredakan batuk?
Bot: Minum air putih hangat 8-10 gelas per hari untuk mengencerkan lendir, minum campuran air hangat, madu, dan lemon efektif meredakan tenggorokan gatal, hirup uap air panas atau gunakan humidifier untuk melegakan pernapasan, minum ramuan herbal,berkumur air garam untuk membantu mengurangi lendir dan meredakan iritasi tenggorokan, kurangi makanan berminyak/pedas, serta hindari rokok dan debu.
Anda: exit
Bot: Terima kasih, semoga sehat selalu!
